# 🧠 Modelo 4: CNN Avanzada con Learning Rate Scheduler

En este notebook utilizamos una arquitectura CNN más robusta (con Batch Normalization) y aplicamos estrategias de **Learning Rate Scheduling** para mejorar la convergencia y evitar el sobreajuste.

---

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub
from sklearn.metrics import classification_report, confusion_matrix

# Añadir el directorio raíz al path para importar módulos locales
sys.path.append('..')
import oct_dataloader as dataloaders
import modelos.modelo_custom as custom_model

print("✅ Librerías e importaciones listas")

In [ ]:
# Configurar GPUs para crecimiento de memoria dinámico
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU(s) detectada(s): {len(gpus)}")
    except RuntimeError as e:
        print(e)
else:
    print("⚠️ No se detectó GPU. Se usará la CPU.")

## 1. Carga de Datos
Utilizamos un subset del 30% del dataset original para acelerar los experimentos, manteniendo la resolución de 128x128.

In [ ]:
# Descargar/Localizar dataset
path = kagglehub.dataset_download("anirudhcv/labeled-optical-coherence-tomography-oct")
data_path = path
for root, dirs, files in os.walk(path):
    if 'train' in dirs and 'test' in dirs:
        data_path = root
        break

# Configuración
IMG_SIZE = (128, 128)
BATCH_SIZE = 32 # El modelo con Batch Normalization suele preferir batches más pequeños o medianos

train_ds, val_ds, test_ds, class_names = dataloaders.create_oct_dataloaders(
    data_path=data_path,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    train_subset_fraction=0.3, 
    optimize=True
)

## 2. Definición del Modelo y Compilación
Configuramos el **Learning Rate** inicial y añadimos la métrica **AUC** para una mejor evaluación en datos desbalanceados.

In [ ]:
from tensorflow.keras.metrics import AUC

# Hiperparámetros ajustables desde fuera
INITIAL_LR = 0.001
DROPOUT = 0.4

# Crear arquitectura
model = custom_model.create_custom_cnn(
    input_shape=(128, 128, 1), 
    num_classes=4, 
    dropout_rate=DROPOUT
)

# Compilar con métricas adicionales
metrics = [
    'accuracy', 
    AUC(name='auc', multi_label=True)
]

model = custom_model.compile_custom_model(
    model, 
    learning_rate=INITIAL_LR, 
    metrics=metrics
)

model.summary()

## 3. Entrenamiento con Scheduler
Implementamos `ReduceLROnPlateau` para reducir el paso de aprendizaje cuando la pérdida de validación deje de mejorar.

In [ ]:
EPOCHS = 20

# Obtener callbacks (EarlyStopping + Scheduler)
callbacks = custom_model.get_callbacks(
    patience_stop=6, 
    patience_lr=3, 
    factor_lr=0.2
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 4. Visualización y Evaluación

In [ ]:
# Curvas de aprendizaje
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history.history['auc'], label='Train')
plt.plot(history.history['val_auc'], label='Val')
plt.title('AUC')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()

plt.show()

In [ ]:
# Resultados finales en Test
print("📊 Evaluación en el conjunto de Test:")
results = model.evaluate(test_ds)
for i, metric in enumerate(model.metrics_names):
    print(f"{metric.capitalize()}: {results[i]:.4f}")